# BESS Profit Optimization Tool
### Battery Energy Storage System — ML Forecasting + Linear Programming

**Pipeline:**
```
Raw Data → Feature Engineering → XGBoost (Price Forecasting)
                                        ↓
                    PuLP LP Optimizer (Charge/Discharge)
                                        ↓
              Final Schedule Table + Net Profit Output
```

**Dataset:** Hourly electricity load and price data for Spain (Kaggle)  
**Battery:** 5 MW max power · 4-hour duration · 20 MWh capacity · 90% round-trip efficiency  
**Constraint:** Grid demand limit of 4 MW max discharge per hour  
**Logic:** Optimizer always charges at low prices, discharges at high prices (enforced by LP objective)

## STEP 0: Install & Import Libraries

In [ ]:
# Uncomment to install if needed
# !pip install xgboost pulp pandas numpy scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from xgboost import XGBRegressor
from pulp import (
    LpProblem, LpMaximize, LpVariable, lpSum, value,
    PULP_CBC_CMD, LpStatus
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ All libraries imported successfully.')

## STEP 1: Data Loading & Preprocessing

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1A. Load the energy dataset
# ─────────────────────────────────────────────────────────────
energy_df = pd.read_csv('/content/energy_dataset.csv')

# Parse datetime with UTC to silence FutureWarning
energy_df['time'] = pd.to_datetime(energy_df['time'], utc=True)

df = energy_df.copy()
print(f'Energy rows : {len(df):,}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1C. Select relevant columns
# ─────────────────────────────────────────────────────────────
TARGET_PRICE = 'price actual'
TARGET_LOAD  = 'total load actual'

RAW_FEATURES = [
    'total load forecast',
    'price day ahead',
]

df_sel = df[['time', TARGET_PRICE, TARGET_LOAD] + RAW_FEATURES].copy()
df_sel['time'] = pd.to_datetime(df_sel['time'], utc=True)
df_sel = df_sel.sort_values('time').set_index('time')

print(f'Selected shape: {df_sel.shape}')
print(f'Date range: {df_sel.index.min()}  →  {df_sel.index.max()}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1D. Clean missing data
# ─────────────────────────────────────────────────────────────
print('Missing values BEFORE cleaning:')
print(df_sel.isnull().sum()[df_sel.isnull().sum() > 0])

df_sel.ffill(inplace=True)   # Forward-fill keeps time-series continuity
df_sel.dropna(inplace=True)  # Drop any rows still missing (i.e. at the very start)

print(f'\nMissing values AFTER cleaning: {df_sel.isnull().sum().sum()}')
print(f'Clean rows: {len(df_sel):,}')

## STEP 2: Feature Engineering

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2A. Cyclical time features  (sin/cos encoding)
# ─────────────────────────────────────────────────────────────
df_sel['hour_sin'] = np.sin(2 * np.pi * df_sel.index.hour / 24)
df_sel['hour_cos'] = np.cos(2 * np.pi * df_sel.index.hour / 24)
df_sel['dow_sin']  = np.sin(2 * np.pi * df_sel.index.dayofweek / 7)
df_sel['dow_cos']  = np.cos(2 * np.pi * df_sel.index.dayofweek / 7)
df_sel['month_sin']= np.sin(2 * np.pi * df_sel.index.month / 12)
df_sel['month_cos']= np.cos(2 * np.pi * df_sel.index.month / 12)

# ─────────────────────────────────────────────────────────────
# 2B. Lag features  (autoregressive signal for the price model)
# ─────────────────────────────────────────────────────────────
for lag in [1, 2, 3, 24, 48, 72]:
    df_sel[f'price_lag_{lag}h'] = df_sel[TARGET_PRICE].shift(lag)
    df_sel[f'load_lag_{lag}h']  = df_sel[TARGET_LOAD].shift(lag)

# ─────────────────────────────────────────────────────────────
# 2C. Rolling mean features  (trend smoothing)
# ─────────────────────────────────────────────────────────────
for window in [6, 12, 24]:
    df_sel[f'price_roll_{window}h'] = df_sel[TARGET_PRICE].rolling(window).mean()
    df_sel[f'load_roll_{window}h']  = df_sel[TARGET_LOAD].rolling(window).mean()

# Drop rows with NaN introduced by lags/rolling
df_sel.dropna(inplace=True)

print(f'Feature-engineered shape: {df_sel.shape}')
print(f'Total features (excl. targets): {df_sel.shape[1] - 2}')

## STEP 3: ML Model — XGBoost Price & Load Forecasting

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3A. Train / test split  (chronological — NO shuffling)
# ─────────────────────────────────────────────────────────────
X = df_sel.drop(columns=[TARGET_PRICE, TARGET_LOAD])
y = df_sel[[TARGET_PRICE, TARGET_LOAD]]

split = int(len(df_sel) * 0.80)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f'Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows')
print(f'Train ends : {X_train.index[-1]}')
print(f'Test  starts: {X_test.index[0]}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3B. Train XGBoost (multi-output via sklearn API)
# ─────────────────────────────────────────────────────────────
model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',   # fast on large data
)

print('Training XGBoost...')
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print('✅ Training complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3C. Evaluate model performance
# ─────────────────────────────────────────────────────────────
y_pred_arr = model.predict(X_test)
y_pred_df  = pd.DataFrame(y_pred_arr, index=y_test.index, columns=[TARGET_PRICE, TARGET_LOAD])

def eval_metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'\n📊 {label}')
    print(f'   MAE  : {mae:.3f}')
    print(f'   RMSE : {rmse:.3f}')
    print(f'   R²   : {r2:.4f}  ({"GOOD" if r2 > 0.85 else "NEEDS WORK"})')

eval_metrics(y_test[TARGET_PRICE], y_pred_df[TARGET_PRICE], 'Price Actual (€/MWh)')
eval_metrics(y_test[TARGET_LOAD],  y_pred_df[TARGET_LOAD],  'Total Load Actual (MW)')

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3D. Visualize: Actual vs Predicted price (first 7 days of test set)
# ─────────────────────────────────────────────────────────────
preview = 168  # 7 days × 24 hours

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(y_test[TARGET_PRICE].iloc[:preview].values,  label='Actual',    color='steelblue', lw=1.5)
axes[0].plot(y_pred_df[TARGET_PRICE].iloc[:preview].values, label='Predicted', color='orange',   lw=1.5, linestyle='--')
axes[0].set_title('Price Actual vs Predicted (First 7 days of test set)')
axes[0].set_ylabel('€/MWh')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(y_test[TARGET_LOAD].iloc[:preview].values,    label='Actual',    color='steelblue', lw=1.5)
axes[1].plot(y_pred_df[TARGET_LOAD].iloc[:preview].values,  label='Predicted', color='orange',   lw=1.5, linestyle='--')
axes[1].set_title('Load Actual vs Predicted')
axes[1].set_ylabel('MW')
axes[1].set_xlabel('Hour index')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## STEP 4: LP Optimizer — Charge / Discharge Schedule

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4A. Core optimizer — MW units + demand constraint
# ─────────────────────────────────────────────────────────────
def run_bess_optimization(
    predicted_prices  : pd.Series,
    max_power_mw      : float = 5.0,
    duration_hrs      : float = 4.0,
    efficiency        : float = 0.90,
    min_soc_pct       : float = 0.10,
    initial_soc_pct   : float = 0.10,
    demand_limit_mw   : float = 4.0,   # grid absorption cap per hour
) -> dict:
    """
    Linear-programming optimizer for BESS energy arbitrage.

    Units: MW / MWh / €/MWh  (national grid scale)

    Key constraints
    ---------------
    1. Max charge power            : max_power_mw
    2. Max discharge power         : min(max_power_mw, demand_limit_mw)  ← demand constraint
    3. Energy balance              : SoC dynamics with round-trip efficiency
    4. SoC bounds                  : [min_soc_pct, 100%]
    5. No simultaneous charge+dis  : binary mutual-exclusion
    6. Charge low / discharge high : enforced automatically by maximising
                                     revenue_from_discharge - cost_of_charge
    """
    MAX_MWH       = max_power_mw * duration_hrs
    MIN_E         = min_soc_pct * MAX_MWH
    INIT_E        = initial_soc_pct * MAX_MWH
    DISCHARGE_CAP = min(max_power_mw, demand_limit_mw)  # tighter of the two limits
    T             = len(predicted_prices)
    HOURS         = range(T)

    prob = LpProblem('BESS_Arbitrage_MW', LpMaximize)

    p_charge    = LpVariable.dicts('Charge',    HOURS, lowBound=0, upBound=max_power_mw)
    p_discharge = LpVariable.dicts('Discharge', HOURS, lowBound=0, upBound=DISCHARGE_CAP)
    soc         = LpVariable.dicts('SoC',       range(T+1), lowBound=MIN_E, upBound=MAX_MWH)
    b_charge    = LpVariable.dicts('b_chg',     HOURS, cat='Binary')
    b_discharge = LpVariable.dicts('b_dis',     HOURS, cat='Binary')

    # Objective: maximise (discharge revenue - charge cost)
    # This naturally forces: charge at LOW prices, discharge at HIGH prices
    prob += lpSum(
        p_discharge[h] * predicted_prices.iloc[h]
      - p_charge[h]   * predicted_prices.iloc[h]
        for h in HOURS
    ), 'Net_Arbitrage_Profit'

    prob += soc[0] == INIT_E,  'Initial_SoC'
    prob += soc[T] >= MIN_E,   'Final_SoC_Min'

    for h in HOURS:
        prob += soc[h+1] == soc[h] + p_charge[h]*efficiency - p_discharge[h]/efficiency
        prob += p_charge[h]    <= b_charge[h]    * max_power_mw,  f'Chg_Flag_{h}'
        prob += p_discharge[h] <= b_discharge[h] * DISCHARGE_CAP, f'Dis_Flag_{h}'
        prob += b_charge[h] + b_discharge[h] <= 1,                f'No_Simult_{h}'

    prob.solve(PULP_CBC_CMD(msg=0))

    net_profit    = value(prob.objective)
    charge_vals   = [p_charge[h].varValue    or 0.0 for h in HOURS]
    dis_vals      = [p_discharge[h].varValue or 0.0 for h in HOURS]
    soc_vals_mwh  = [soc[h].varValue         or MIN_E for h in range(T+1)]
    soc_vals_pct  = [100*v/MAX_MWH for v in soc_vals_mwh]

    actions = [
        'Charge'    if c > 0.001 else
        'Discharge' if d > 0.001 else
        'Idle'
        for c, d in zip(charge_vals, dis_vals)
    ]

    schedule_df = pd.DataFrame({
        'Hour'           : list(range(T)),
        'Timestamp'      : predicted_prices.index,
        'Price (€/MWh)'  : predicted_prices.values.round(2),
        'Action'         : actions,
        'Charge (MW)'    : [round(v, 3) for v in charge_vals],
        'Discharge (MW)' : [round(v, 3) for v in dis_vals],
        'SoC Start (%)'  : [round(v, 1) for v in soc_vals_pct[:T]],
        'SoC End (%)'    : [round(v, 1) for v in soc_vals_pct[1:]],
        'Revenue (€)'    : [round(d*p, 2) for d, p in zip(dis_vals, predicted_prices.values)],
        'Cost (€)'       : [round(c*p, 2) for c, p in zip(charge_vals, predicted_prices.values)],
    })
    schedule_df['Net P&L (€)'] = (schedule_df['Revenue (€)'] - schedule_df['Cost (€)']).round(2)

    soc_series = pd.Series(soc_vals_pct, name='SoC (%)')

    return {
        'total_profit'   : net_profit,
        'schedule_df'    : schedule_df,
        'soc_series'     : soc_series,
        'status'         : LpStatus[prob.status],
        'max_mwh'        : MAX_MWH,
        'discharge_cap'  : DISCHARGE_CAP,
    }

print('run_bess_optimization() defined — MW units, demand constraint active.')


## STEP 5: Connect ML → Optimizer (The Full Pipeline)

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5A. Pick a target day from the TEST SET
#     (using ML-predicted prices — NOT actual prices)
# ─────────────────────────────────────────────────────────────

# Use the first full 24-hour window from the test predictions
# These are the prices the model PREDICTED — this is the key pipeline connection
predicted_prices_24h = y_pred_df[TARGET_PRICE].iloc[:24]

print('📅 Optimization window:')
print(f'   Start : {predicted_prices_24h.index[0]}')
print(f'   End   : {predicted_prices_24h.index[-1]}')
print(f'\n   Min predicted price : {predicted_prices_24h.min():.2f} €/MWh')
print(f'   Max predicted price : {predicted_prices_24h.max():.2f} €/MWh')
print(f'   Price spread        : {predicted_prices_24h.max() - predicted_prices_24h.min():.2f} €/MWh')

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5B. Battery configuration — MW scale + demand constraint
# ─────────────────────────────────────────────────────────────
BATTERY_CONFIG = dict(
    max_power_mw      = 5.0,   # MW  (max charge/discharge rate)
    duration_hrs      = 4.0,   # hours → capacity = 5 × 4 = 20 MWh
    efficiency        = 0.90,  # 90% round-trip efficiency
    min_soc_pct       = 0.10,  # 10% minimum SoC
    initial_soc_pct   = 0.10,  # battery starts at 10%
    demand_limit_mw   = 4.0,   # grid can only absorb 4 MW at a time
)

print('Battery Configuration (MW scale)')
for k, v in BATTERY_CONFIG.items():
    print(f'   {k:<20}: {v}')
cap = BATTERY_CONFIG['max_power_mw'] * BATTERY_CONFIG['duration_hrs']
print(f'   {"capacity (MWh)":<20}: {cap}')
print(f'   {"discharge_cap (MW)":<20}: {min(BATTERY_CONFIG["max_power_mw"], BATTERY_CONFIG["demand_limit_mw"])}')


In [ ]:
# ─────────────────────────────────────────────────────────────
# 5C. Run the optimizer on ML-predicted prices
# ─────────────────────────────────────────────────────────────
result = run_bess_optimization(predicted_prices_24h, **BATTERY_CONFIG)

print(f'\n🔑 Optimizer Status : {result["status"]}')
print(f'💰 Maximum Net Arbitrage Profit : {result["total_profit"]:.4f} €')

## STEP 6: Final Output — Schedule Table & Profit Summary

In [ ]:
# ─────────────────────────────────────────────────────────────
# 6A. 24-Hour Optimal Schedule Table
# ─────────────────────────────────────────────────────────────
schedule = result['schedule_df']

print('=' * 90)
print('             24-HOUR OPTIMAL BESS SCHEDULE (ML-Predicted Prices)')
print('=' * 90)
display(schedule.style
    .format({
        'Price (€/MWh)'  : '{:.2f}',
        'Charge (MW)'    : '{:.3f}',
        'Discharge (MW)' : '{:.3f}',
        'SoC Start (%)'  : '{:.1f}',
        'SoC End (%)'    : '{:.1f}',
        'Revenue (€)'    : '{:.4f}',
        'Cost (€)'       : '{:.4f}',
        'Net P&L (€)'    : '{:.4f}',
    })
    .applymap(lambda v: 'background-color: #d4edda' if v == '⚡ Charge'
              else ('background-color: #f8d7da' if v == '💰 Discharge' else ''),
              subset=['Action'])
    .bar(subset=['Net P&L (€)'], color=['#f8d7da', '#d4edda'], align='zero')
)

In [ ]:
# ─────────────────────────────────────────────────────────────
# 6B. Profit Summary
# ─────────────────────────────────────────────────────────────
total_revenue     = schedule['Revenue (€)'].sum()
total_cost        = schedule['Cost (€)'].sum()
net_profit        = result['total_profit']
n_charge_hours    = (schedule['Action'] == 'Charge').sum()
n_discharge_hours = (schedule['Action'] == 'Discharge').sum()
n_idle_hours      = (schedule['Action'] == 'Idle').sum()
peak_soc          = schedule['SoC End (%)'].max()

# Verify charge-low / discharge-high logic
charge_prices    = schedule[schedule['Action']=='Charge']['Price (€/MWh)']
discharge_prices = schedule[schedule['Action']=='Discharge']['Price (€/MWh)']

print('=' * 60)
print('              BESS PROFIT SUMMARY')
print('=' * 60)
print(f'  Gross Revenue (discharge)    : €{total_revenue:>10,.2f}')
print(f'  Gross Cost    (charge)       : €{total_cost:>10,.2f}')
print(f'  ─────────────────────────────────────────')
print(f'  NET ARBITRAGE PROFIT         : €{net_profit:>10,.2f}')
print('=' * 60)
print(f'\n  Schedule breakdown (24h):')
print(f'  Charge hours     : {n_charge_hours}')
print(f'  Discharge hours  : {n_discharge_hours}')
print(f'  Idle hours       : {n_idle_hours}')
print(f'  Peak SoC reached : {peak_soc:.1f}%')
print(f'  Battery capacity : {result["max_mwh"]:.1f} MWh')
print(f'  Discharge cap    : {result["discharge_cap"]:.1f} MW  (demand constraint)')
print(f'\n  Charge/Discharge price check:')
if len(charge_prices) and len(discharge_prices):
    print(f'  Avg charge price    : €{charge_prices.mean():.2f}/MWh')
    print(f'  Avg discharge price : €{discharge_prices.mean():.2f}/MWh')
    ok = charge_prices.mean() < discharge_prices.mean()
    print(f'  Logic OK (charge < discharge): {ok}')


## STEP 7: Visualization — Price + Charge/Discharge + SoC

In [ ]:
# ─────────────────────────────────────────────────────────────
# 7A. Main optimization chart  (dual Y-axis)
# ─────────────────────────────────────────────────────────────
fig, (ax1, ax3) = plt.subplots(2, 1, figsize=(16, 10), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

hours = range(len(schedule))
prices = schedule['Price (€/MWh)'].values
charge = schedule['Charge (MW)'].values
discharge = schedule['Discharge (MW)'].values
soc_start = schedule['SoC Start (%)'].values
soc_end   = schedule['SoC End (%)'].values

# ── Top panel: Price + Charge/Discharge bars ─────────────────
ax2 = ax1.twinx()

ax1.plot(hours, prices, color='#2196F3', lw=2, zorder=5, label='Predicted Price (€/MWh)')
ax1.set_ylabel('Electricity Price (€/MWh)', color='#2196F3', fontsize=11)
ax1.tick_params(axis='y', labelcolor='#2196F3')
ax1.grid(True, linestyle='--', alpha=0.3)

ax2.bar(hours, charge,    color='#4CAF50', alpha=0.75, width=0.7, label='Charge (MW)')
ax2.bar(hours, -discharge, color='#F44336', alpha=0.75, width=0.7, label='Discharge (MW)')
ax2.set_ylabel('Power (MW)  [+ = Charge, – = Discharge]', color='gray', fontsize=10)
ax2.axhline(0, color='black', linewidth=0.8)

# Combine legends
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=9)
ax1.set_title('BESS Optimization: Predicted Price & Charge/Discharge Schedule', fontsize=13, fontweight='bold')

# ── Bottom panel: SoC % ──────────────────────────────────────
soc_plot = list(soc_start) + [soc_end[-1]]
ax3.step(range(len(soc_plot)), soc_plot, where='post', color='#9C27B0', lw=2, label='SoC (%)')
ax3.fill_between(range(len(soc_plot)), soc_plot, step='post', alpha=0.2, color='#9C27B0')
ax3.axhline(BATTERY_CONFIG['min_soc_pct'] * 100, color='red', linestyle=':', lw=1.2, label=f'Min SoC ({BATTERY_CONFIG["min_soc_pct"]*100:.0f}%)')
ax3.set_ylabel('State of Charge (%)', fontsize=10)
ax3.set_xlabel('Hour of Day', fontsize=10)
ax3.set_ylim(0, 105)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

# X-axis labels: show actual timestamps
tick_labels = [str(schedule['Timestamp'].iloc[h].strftime('%H:%M')) if h % 4 == 0 else '' for h in hours]
ax3.set_xticks(hours)
ax3.set_xticklabels(tick_labels, rotation=30)

plt.tight_layout()
plt.savefig('bess_schedule.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📊 Chart saved as bess_schedule.png')

In [ ]:
# ─────────────────────────────────────────────────────────────
# 7B. Feature importance (what drives the price forecast)
# ─────────────────────────────────────────────────────────────
feat_imp = pd.Series(model.feature_importances_, index=X_train.columns)
top20 = feat_imp.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 6))
top20.sort_values().plot(kind='barh', ax=ax, color='steelblue', alpha=0.8)
ax.set_title('Top 20 Feature Importances (XGBoost)', fontsize=12)
ax.set_xlabel('Importance Score')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## STEP 8: Save Artifacts (Model + Schedule)

Save the trained model and the final schedule so they can be loaded by the Streamlit app without retraining.

In [ ]:
import joblib

# Save the trained model
joblib.dump(model, 'bess_xgboost_model.pkl')
print('✅ Model saved → bess_xgboost_model.pkl')

# Save feature column list (needed for Streamlit inference)
joblib.dump(list(X_train.columns), 'bess_feature_columns.pkl')
print('✅ Feature columns saved → bess_feature_columns.pkl')

# Save the last schedule as CSV for reference
schedule.to_csv('bess_last_schedule.csv')
print('✅ Schedule saved → bess_last_schedule.csv')

---
## ✅ Pipeline Complete

| Step | What it does |
|------|--------------|
| 1 | Load Spain energy dataset (load + price data) |
| 2 | Feature engineering (cyclical, lags, rolling means) |
| 3 | XGBoost trains on 80% of data to predict `price actual` |
| 4 | LP optimizer (`run_bess_optimization`) formulates and solves the charge/discharge problem |
| 5 | **Bridge**: ML-predicted prices → fed directly into optimizer |
| 6 | Final 24-hour schedule table + net profit |
| 7 | Visualizations |
| 8 | Artifacts saved (model + schedule) for Streamlit |

**Next:** Run `streamlit run bess_app.py` to launch the interactive dashboard.